#Installation

In [2]:
!apt-get -qq update
!apt-get -qq install -y libatk1.0-0 libatk-bridge2.0-0 libcups2 libxkbcommon0 libxcomposite1 libxrandr2 libxdamage1 libpangocairo-1.0-0 libasound2 libnss3 libgbm1
!ldconfig -p | grep libatk-1.0.so.0 || echo "libatk NOT found"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../0-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../1-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../2-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../3-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubuntu1) ...
Selecting previously unselected pac

In [4]:
!pip -q install playwright bs4 lxml pandas
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 21.2 MB/s eta 0:00:00
(node:1911) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
164.7 MiB [] 0% 334.7s164.7 MiB [] 0% 116.1s164.7 MiB [] 0% 386.8s164.7 MiB [] 0% 436.5s164.7 MiB [] 0% 349.3s164.7 MiB [] 0% 302.0s164.7 MiB [] 0% 271.0s164.7 MiB [] 0% 250.3s164.7 MiB [] 0% 234.4s164.7 MiB [] 0% 219.2s164.7 MiB [] 0% 206.6s164.7 MiB [] 0% 194.9s164.7 MiB [] 0% 170.8s164.7 MiB [] 0% 153.7s164.7 MiB [] 0% 140.8s164.7 MiB [] 0% 130.6s164.7 MiB [] 0% 124.7s164.7 MiB [] 0% 115.0s164.7 MiB [] 0% 104.7s164.7 MiB [] 0% 94.9s164.7 MiB [] 0% 85.5s164.7 MiB [] 0% 77.4s164.7 MiB [] 0% 72.0s164.7 MiB [] 0% 66.0s164.7 MiB [] 0% 60.6s164.7 MiB [] 1% 55.2s164.7 MiB [] 1% 50.1s164.7 MiB [] 1% 45.5s164.7 MiB [] 1

In [5]:
!ldconfig -p | grep libatk-1.0.so.0 || echo "libatk NOT found"


	libatk-1.0.so.0 (libc6,x86-64) => /lib/x86_64-linux-gnu/libatk-1.0.so.0


#Scrapping : scroll + clic carrousel + extraction

In [6]:
import re, json, asyncio, random
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


BASE_URL = "https://www.movieofthenight.com"
CATALOG_URL = "https://www.movieofthenight.com/catalog/fr"


# =============================
# DATA MODELS
# =============================
@dataclass
class PlatformOffer:
    provider: Optional[str]
    offer: Optional[str]
    quality: Optional[str]
    price_or_note: Optional[str]
    url: Optional[str]

@dataclass
class ShowData:
    show_id: int
    show_url: str
    title: Optional[str]
    poster_url: Optional[str]
    rating: Optional[float]
    year: Optional[int]
    tags: List[str]
    duration_minutes: Optional[int]
    summary: Optional[str]
    director: Optional[str]
    starring: Optional[str]
    platforms: List[PlatformOffer]


# =============================
# PARSING HELPERS (SHOW PAGE)
# =============================
def parse_year_genres_duration(meta_line: str):
    parts = [p.strip() for p in meta_line.split("•")]
    year, tags, duration_min = None, [], None

    if parts and re.fullmatch(r"\d{4}", parts[0]):
        year = int(parts[0])

    if len(parts) >= 2:
        tags = [g.strip() for g in parts[1].split(",") if g.strip()]

    if len(parts) >= 3:
        h = int(re.search(r"(\d+)\s*Hour", parts[2]).group(1)) if "Hour" in parts[2] else 0
        m = int(re.search(r"(\d+)\s*Minute", parts[2]).group(1)) if "Minute" in parts[2] else 0
        duration_min = h * 60 + m if (h or m) else None

    return year, tags, duration_min


def extract_labeled_value(soup: BeautifulSoup, label: str) -> Optional[str]:
    el = soup.find(string=lambda s: isinstance(s, str) and s.strip() == label)
    if not el:
        return None
    parent = el.parent.parent
    texts = list(parent.stripped_strings)
    for i, t in enumerate(texts):
        if t == label and i + 1 < len(texts):
            return texts[i + 1]
    return None


def extract_summary(soup: BeautifulSoup) -> Optional[str]:
    texts = list(soup.stripped_strings)
    stop = texts.index("How to Watch?") if "How to Watch?" in texts else len(texts)
    long_texts = [t for t in texts[:stop] if len(t) > 120]
    return max(long_texts, key=len) if long_texts else None


def extract_poster_url(soup: BeautifulSoup) -> Optional[str]:
    for img in soup.find_all("img", src=True):
        if "image.tmdb.org" in img["src"]:
            return img["src"]
    return None


def first_rating_near_title(soup: BeautifulSoup, title: str) -> Optional[float]:
    texts = list(soup.stripped_strings)
    try:
        i = texts.index(title)
    except ValueError:
        return None

    for t in texts[i:i+200]:
        if re.fullmatch(r"\d\.\d", t):
            return float(t)
    return None


def extract_provider_from_logo(src: Optional[str]) -> Optional[str]:
    if not src:
        return None
    m = re.search(r"/services/([^/]+)/", src)
    if not m:
        return None
    slug = m.group(1).lower()
    mapping = {
        "prime": "Prime Video",
        "disney": "Disney+",
        "netflix": "Netflix",
        "appletv": "Apple TV",
        "paramount": "Paramount+",
        "hbo": "HBO Max",
        "mubi": "Mubi",
        "canal": "CANAL+",
        "ocs": "OCS",
    }
    return mapping.get(slug, slug.capitalize())


def extract_platforms(soup: BeautifulSoup) -> List[PlatformOffer]:
    offers = []
    how = soup.find(string=lambda s: s and s.strip() == "How to Watch?")
    if not how:
        return offers

    stop_labels = {"Director", "Starring", "Trailer"}
    seen = set()

    for a in how.parent.find_all_next("a", href=True):
        if a.get_text(strip=True) in stop_labels:
            break

        href = a["href"]
        texts = list(a.stripped_strings)

        offer = next((t for t in texts if "on" in t or "via" in t), None)
        quality = next((t for t in texts if t in {"SD", "HD", "UHD", "4K"}), None)
        price = next((t for t in texts if "EUR" in t or "Subscription" in t), None)

        img = a.find("img")
        provider = extract_provider_from_logo(img["src"] if img else None)

        key = (provider, href)
        if key in seen:
            continue
        seen.add(key)

        offers.append(PlatformOffer(provider, offer, quality, price, href))

    return offers


def parse_show_html(html: str, url: str) -> Optional[ShowData]:
    soup = BeautifulSoup(html, "lxml")
    m = re.search(r"/show/(\d+)", url)
    if not m:
        return None
    show_id = int(m.group(1))

    title_el = soup.find("h1")
    if not title_el:
        return None

    title = title_el.get_text(strip=True)
    meta = title_el.find_next(string=lambda s: "•" in s)
    year, tags, duration = parse_year_genres_duration(meta) if meta else (None, [], None)

    return ShowData(
        show_id=show_id,
        show_url=url.split("?")[0],
        title=title,
        poster_url=extract_poster_url(soup),
        rating=first_rating_near_title(soup, title),
        year=year,
        tags=tags,
        duration_minutes=duration,
        summary=extract_summary(soup),
        director=extract_labeled_value(soup, "Director"),
        starring=extract_labeled_value(soup, "Starring"),
        platforms=extract_platforms(soup),
    )


# =============================
# CATALOG COLLECTOR
# =============================
async def collect_catalog_links(page, max_scrolls=200):
    await page.goto(CATALOG_URL, wait_until="domcontentloaded")
    await asyncio.sleep(2)

    links = set()
    for _ in range(max_scrolls):
        soup = BeautifulSoup(await page.content(), "lxml")
        for a in soup.find_all("a", href=True):
            if re.match(r"^/show/\d+", a["href"]):
                links.add(urljoin(BASE_URL, a["href"].split("?")[0]))
        await page.mouse.wheel(0, 3000)
        await asyncio.sleep(random.uniform(0.6, 1.2))

    return sorted(links)


# =============================
# MAIN RUNNER
# =============================
async def run_scraper(concurrency=3):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        context = await browser.new_context(locale="fr-FR")

        page = await context.new_page()
        print("[1/3] Collecting catalog…")
        links = await collect_catalog_links(page)
        await page.close()
        print(f"Found {len(links)} shows")

        sem = asyncio.Semaphore(concurrency)
        results = []

        async def scrape_one(url):
            async with sem:
                page = await context.new_page()
                try:
                    await page.goto(url + "?id=1", wait_until="domcontentloaded")
                    await page.wait_for_selector("h1", timeout=12000)
                    data = parse_show_html(await page.content(), url)
                    if data:
                        results.append(asdict(data))
                finally:
                    await page.close()
                    await asyncio.sleep(random.uniform(0.4, 0.9))

        await asyncio.gather(*(scrape_one(u) for u in links))
        await browser.close()

    print("[3/3] Exporting…")

    rows = []
    for d in results:
        providers = sorted({p["provider"] for p in d["platforms"] if p["provider"]})
        d["platforms"] = "; ".join(providers)
        d["tags"] = ", ".join(d["tags"])
        rows.append(d)

    pd.DataFrame(rows).to_csv(
        "movieofthenight_catalog_fr.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Done. rows={len(rows)}")
    return rows


# ▶ RUN
data = await run_scraper(concurrency=3)
print("Example:", data[0]["title"] if data else None)


[1/3] Collecting catalog…
Found 100 shows
[3/3] Exporting…
Done. rows=100
Example: The Silence of the Lambs
